# Notebook 02 — Functions, Limits, and Derivatives

**Module:** 02 — Mathematics for Biology  
**Notebook:** 02 of 12  
**Prerequisites:** NB01  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Every ODE in biology contains a derivative. `dN/dt` is not notation — it is the
instantaneous rate of change of population size. To write an ODE model, you must
know what a derivative is, how to compute one, and what it means when it is zero,
positive, or negative.

---
## Step 2 — Intuition

A derivative answers: *how fast is this quantity changing right now?*

- Bacterial population: $dN/dt > 0$ → growing; $dN/dt < 0$ → dying
- Drug concentration: $dC/dt < 0$ → being cleared
- mRNA level: $dM/dt = 0$ → at steady state

The 'limit' definition makes this rigorous: as the time interval shrinks to zero,
the average rate of change becomes the instantaneous rate.

---
## Step 3 — Biological Background

**mRNA production and degradation:**

A gene produces mRNA at rate $\alpha$ (transcription). mRNA degrades at rate
$\delta M$ (first-order degradation — the more mRNA, the faster it degrades).

$$\frac{dM}{dt} = \alpha - \delta M$$

At steady state, $dM/dt = 0$, so $M^* = \alpha/\delta$.

This is the simplest meaningful biological ODE. Every symbol has a physical meaning:
- $\alpha$: transcription rate (mRNA molecules per minute)
- $\delta$: degradation rate (per minute — so $\delta M$ has units mRNA/min)
- $M^*$: steady-state mRNA level

---
## Step 4 — Mathematical Explanation

**Formal definition of the derivative:**

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

**Key derivatives for biology:**

| Function | Derivative | Biological appearance |
|----------|-----------|----------------------|
| $f(t) = e^{rt}$ | $f'(t) = r e^{rt}$ | Exponential growth: $dN/dt = rN$ |
| $f(t) = \ln(t)$ | $f'(t) = 1/t$ | Log-transformed growth rates |
| $f(t) = t^n$ | $f'(t) = n t^{n-1}$ | Polynomial approximations |
| $f(t) = \frac{1}{1+e^{-t}}$ | $f'(t) = f(t)(1-f(t))$ | Logistic sigmoid; Hill function |

**Numerical derivative (central difference):**

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

Error is $O(h^2)$ — much better than the forward difference $O(h)$.

---
## Step 5 — Computational Explanation

**Three ways to compute a derivative in Python:**

1. **Analytically** — by hand or with `sympy`: exact, no numerical error
2. **Numerically** — central difference: fast, small error with small `h`
3. **Automatically** — `jax.grad` or `torch.autograd`: exact gradients for arbitrary code

This module uses method 2. Method 3 is covered in Module 13 (Machine Learning).

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Central difference numerical derivative
def numerical_derivative(f, x: float, h: float = 1e-5) -> float:
    """
    Compute df/dx at x using the central difference formula.

    Parameters
    ----------
    f : callable
        The function to differentiate.
    x : float
        Point at which to evaluate the derivative.
    h : float
        Step size (default 1e-5).

    Returns
    -------
    float
        Numerical approximation of f'(x).
    """
    return (f(x + h) - f(x - h)) / (2 * h)

# Verify on known functions
print("Numerical vs analytical derivatives:")
print(f"  d/dx[e^x] at x=1:  numerical={numerical_derivative(np.exp, 1):.8f}  analytical={np.e:.8f}")
print(f"  d/dx[ln x] at x=2: numerical={numerical_derivative(np.log, 2):.8f}  analytical={0.5:.8f}")
print(f"  d/dx[x^3] at x=3:  numerical={numerical_derivative(lambda x: x**3, 3):.8f}  analytical={27:.8f}")

In [ ]:
# Cell 6.2 — Derivative of growth rate: instantaneous vs average
# E. coli: N(t) = N0 * e^(r*t), r = ln(2) / T_double
N0, T_double = 1, 20  # cells, minutes
r = np.log(2) / T_double
N = lambda t: N0 * np.exp(r * t)
dN_dt = lambda t: r * N0 * np.exp(r * t)  # analytical derivative

t_vals = np.array([0, 20, 40, 60, 80, 100])
print("E. coli: population and growth rate")
print(f"{'t (min)':<10} {'N(t)':<12} {'dN/dt (numerical)':<22} {'dN/dt (analytical)'}")
print("-" * 65)
for t in t_vals:
    num = numerical_derivative(N, t)
    ana = dN_dt(t)
    print(f"  {t:<10} {N(t):<12.2f} {num:<22.4f} {ana:.4f}")

In [ ]:
# Cell 6.3 — mRNA ODE: steady state from setting dM/dt = 0
alpha = 5.0   # mRNA molecules / min
delta = 0.5   # per min
M_star = alpha / delta  # steady-state: dM/dt = 0 → M* = alpha/delta

print(f"mRNA ODE: dM/dt = {alpha} - {delta}·M")
print(f"Steady state M* = alpha/delta = {alpha}/{delta} = {M_star} molecules")
print()

# Check: at M*, what is dM/dt?
dM_dt = lambda M: alpha - delta * M
print(f"dM/dt at M*:  {dM_dt(M_star):.6f}  (should be 0)")
print(f"dM/dt at M=5: {dM_dt(5):.2f}    (positive: moving toward M*)")
print(f"dM/dt at M=15:{dM_dt(15):.2f}   (negative: moving toward M*)")

In [ ]:
# Cell 6.4 — Error analysis: how does step size h affect accuracy?
h_values = np.logspace(-1, -12, 50)
f = np.exp
true_deriv = np.e  # d/dx[e^x] at x=1 = e

errors = [abs(numerical_derivative(f, 1.0, h=h) - true_deriv) for h in h_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_values, errors, color="steelblue", lw=1.5)
ax.set_xlabel("Step size h")
ax.set_ylabel("|error|")
ax.set_title("Central difference error vs. step size for d/dx[e^x] at x=1")
ax.invert_xaxis()
# Mark optimal h
opt_h = h_values[np.argmin(errors)]
ax.axvline(opt_h, color="tomato", ls="--", label=f"Optimal h ≈ {opt_h:.0e}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Optimal step size: h ≈ {opt_h:.0e}  (beyond this, floating-point cancellation dominates)")

---
## Step 7 — Visualization (included in Cell 6.4)

In [ ]:
# Cell 7.1 — Tangent line visualization: derivative as slope
t_fine = np.linspace(0, 3, 300)
N_curve = N0 * np.exp(r * t_fine)

# Tangent at t=40 min
t_tan = 40
slope = numerical_derivative(N, t_tan)
tangent = lambda t: N(t_tan) + slope * (t - t_tan)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_fine, N_curve, color="steelblue", lw=2, label="N(t) = e^{rt}")
t_range = np.linspace(25, 55, 50)
ax.plot(t_range, tangent(t_range), color="tomato", lw=1.5, ls="--",
        label=f"Tangent at t=40: slope={slope:.2f} cells/min")
ax.scatter([t_tan], [N(t_tan)], color="black", zorder=5)
ax.set_xlabel("Time (min)"); ax.set_ylabel("N (cells)")
ax.set_title("Derivative = slope of tangent line")
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `numerical_derivative` and verify it on: `f(x) = sin(x)` at `x = π/4`
   (analytical: `cos(π/4) = √2/2`). Report the error.
2. The Hill function is $f(x) = x^n / (K^n + x^n)$. Compute its derivative numerically
   at `x = K` for `n = 1, 2, 4`. What happens as `n` increases?
3. For the mRNA ODE: set `alpha = 10`, `delta = 0.2`. Without running code, predict
   $M^*$. Verify by running.
4. Why does the central difference error increase again for very small `h`
   (visible in Cell 6.4's plot)? Write a one-sentence explanation.

---
## Quiz — Active Recall

1. What does $dM/dt = 0$ mean biologically for an mRNA molecule?
2. Write the central difference formula from memory.
3. Why is the central difference more accurate than the forward difference?
4. If $N(t) = e^{rt}$, what is $dN/dt$? No looking.
5. What is $M^* = \alpha/\delta$ in the mRNA ODE? Derive it in one line.

---
## Reflection

**Date completed:** ____________________

> *[Can you explain the tangent line interpretation of a derivative in plain language? What was the most confusing part?]*

---
*Next: `03_integrals_and_auc.ipynb`*